# Module 10: Normalization in Deep Networks

## Why Normalization Matters

Training deep neural networks is fundamentally difficult. As data flows through many layers,
the distribution of activations can shift dramatically -- a phenomenon known as **internal covariate shift**.
Each layer must constantly adapt to the changing distribution of its inputs, which slows training
and can cause gradient explosion or vanishing.

**Normalization layers** address this by re-centering and re-scaling activations at each layer,
keeping them in a well-behaved range. This leads to:

- **Faster convergence**: gradients flow more smoothly through the network
- **Higher learning rates**: normalized activations are less sensitive to scale
- **Training stability**: prevents activations from growing unboundedly

In this notebook we build **Layer Normalization** from scratch, compare it to Batch Normalization,
explore RMSNorm, and demonstrate why normalization is essential for transformer architectures.

## 1. Imports and Setup

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from layernorm import LayerNorm

torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Manual LayerNorm: Step by Step

Before using any module, let us walk through the LayerNorm formula manually.

Given an input vector $x$ of dimension $d$:

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

$$y_i = \gamma \cdot \hat{x}_i + \beta$$

Where $\mu$ is the mean, $\sigma^2$ is the variance, $\gamma$ is a learnable scale, and $\beta$ is a learnable shift.

In [ ]:
# Create a small input to trace through each step
torch.manual_seed(42)
x = torch.randn(1, 8)  # batch=1, features=8
print("Input x:")
print(x)
print(f"  shape: {x.shape}")

In [ ]:
# Step 1: Compute the mean across the last dimension
mean = x.mean(dim=-1, keepdim=True)
print(f"Mean: {mean.item():.6f}")

# Step 2: Compute the variance across the last dimension
var = x.var(dim=-1, keepdim=True)
print(f"Variance: {var.item():.6f}")

# Step 3: Normalize (subtract mean, divide by std)
eps = 1e-5
x_normalized = (x - mean) / torch.sqrt(var + eps)
print(f"\nNormalized x:")
print(x_normalized)
print(f"  mean after normalization: {x_normalized.mean().item():.8f}")
print(f"  var  after normalization: {x_normalized.var().item():.6f}")

In [ ]:
# Step 4: Apply learnable scale (gamma) and shift (beta)
gamma = torch.ones(8)   # initialized to 1 -- identity scale
beta = torch.zeros(8)   # initialized to 0 -- no shift

y = gamma * x_normalized + beta
print("Output y (with default gamma=1, beta=0):")
print(y)
print(f"  Same as x_normalized? {torch.allclose(y, x_normalized)}")

# With non-trivial gamma and beta, the output distribution shifts
gamma_custom = torch.full((8,), 2.0)
beta_custom = torch.full((8,), 0.5)
y_custom = gamma_custom * x_normalized + beta_custom
print(f"\nWith gamma=2, beta=0.5:")
print(f"  mean: {y_custom.mean().item():.6f}")
print(f"  std:  {y_custom.std().item():.6f}")

## 3. Using Our Custom LayerNorm Module

Now let us use the `LayerNorm` class from `layernorm.py` on a larger input and verify it normalizes correctly.

In [ ]:
torch.manual_seed(42)

d = 64
ln = LayerNorm(d)

# Random input: batch of 4 vectors, each of dimension 64
x = torch.randn(4, d) * 5 + 3  # deliberately shifted and scaled

print("=== Before LayerNorm ===")
print(f"  mean per sample: {x.mean(dim=-1).tolist()}")
print(f"  var  per sample: {x.var(dim=-1).tolist()}")

y = ln(x)

print("\n=== After LayerNorm ===")
print(f"  mean per sample: {[f'{m:.6f}' for m in y.mean(dim=-1).tolist()]}")
print(f"  var  per sample: {[f'{v:.4f}' for v in y.var(dim=-1).tolist()]}")
print("\nMean is approximately 0 and variance is approximately 1 for each sample.")

## 4. Verify Against PyTorch's nn.LayerNorm

Let us confirm our implementation matches PyTorch's built-in `nn.LayerNorm`.

In [ ]:
torch.manual_seed(42)

d = 64
x = torch.randn(4, d)

# Our custom LayerNorm
our_ln = LayerNorm(d)
our_output = our_ln(x)

# PyTorch's built-in LayerNorm
pt_ln = nn.LayerNorm(d)
# Both start with gamma=1, beta=0 by default
pt_output = pt_ln(x)

# Compare
max_diff = (our_output - pt_output).abs().max().item()
match = torch.allclose(our_output, pt_output, atol=1e-6)

print(f"Max absolute difference: {max_diff:.2e}")
print(f"Outputs match (atol=1e-6): {match}")

# Show a few values side by side
print("\nSample values (first 5 features of first sample):")
print(f"  Ours:    {our_output[0, :5].tolist()}")
print(f"  PyTorch: {pt_output[0, :5].tolist()}")

## 5. Visualize: Input vs. Output Distributions

A histogram makes it clear what LayerNorm does -- it centers and rescales the activations.

In [ ]:
torch.manual_seed(42)

# Create input with non-standard distribution
x = torch.randn(32, 128) * 3 + 5  # mean~5, std~3
ln = LayerNorm(128)
y = ln(x)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(x.detach().numpy().flatten(), bins=60, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('Before LayerNorm')
axes[0].set_xlabel('Activation value')
axes[0].set_ylabel('Count')
axes[0].axvline(x.mean().item(), color='red', linestyle='--', label=f'mean={x.mean().item():.2f}')
axes[0].legend()

axes[1].hist(y.detach().numpy().flatten(), bins=60, alpha=0.7, color='coral', edgecolor='black', linewidth=0.5)
axes[1].set_title('After LayerNorm')
axes[1].set_xlabel('Activation value')
axes[1].set_ylabel('Count')
axes[1].axvline(y.mean().item(), color='red', linestyle='--', label=f'mean={y.mean().item():.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Overlaid histograms for direct comparison
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(x.detach().numpy().flatten(), bins=60, alpha=0.5, color='steelblue', label='Before LayerNorm', density=True)
ax.hist(y.detach().numpy().flatten(), bins=60, alpha=0.5, color='coral', label='After LayerNorm', density=True)
ax.set_xlabel('Activation value')
ax.set_ylabel('Density')
ax.set_title('Input vs. Output Distribution (Overlaid)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Learnable Parameters: Gamma and Beta

After normalization, the output has mean 0 and variance 1. But this might be too restrictive --
the network may *need* a different distribution at certain layers.

The learnable parameters $\gamma$ (gain/scale) and $\beta$ (bias/shift) let the network
recover any mean and variance it wants. At initialization:
- $\gamma = 1$ (identity scale)
- $\beta = 0$ (no shift)

So initially LayerNorm just normalizes, but during training the network can learn
to undo the normalization if that helps.

In [ ]:
ln = LayerNorm(64)

print("Gamma (gain) -- self.g:")
print(f"  shape: {ln.g.shape}")
print(f"  values (first 10): {ln.g.data[:10].tolist()}")
print(f"  requires_grad: {ln.g.requires_grad}")

print("\nBeta (bias) -- self.b:")
print(f"  shape: {ln.b.shape}")
print(f"  values (first 10): {ln.b.data[:10].tolist()}")
print(f"  requires_grad: {ln.b.requires_grad}")

print("\nTotal learnable parameters:", sum(p.numel() for p in ln.parameters()))

In [ ]:
# Demonstrate that modified gamma/beta change the output distribution
torch.manual_seed(42)
x = torch.randn(16, 64)

# Default: gamma=1, beta=0
y_default = ln(x)

# Modify parameters: gamma=3, beta=10
with torch.no_grad():
    ln.g.fill_(3.0)
    ln.b.fill_(10.0)

y_modified = ln(x)

print("Default gamma=1, beta=0:")
print(f"  output mean: {y_default.mean().item():.4f}, std: {y_default.std().item():.4f}")

print("\nModified gamma=3, beta=10:")
print(f"  output mean: {y_modified.mean().item():.4f}, std: {y_modified.std().item():.4f}")
print("\nThe network can learn to shift and scale the normalized output as needed.")

## 7. LayerNorm vs. BatchNorm

The key difference is **which dimension** is normalized:

| Method | Normalizes across | Use case |
|--------|------------------|----------|
| **BatchNorm** | Batch dimension (for each feature independently) | CNNs, fixed batch sizes |
| **LayerNorm** | Feature dimension (for each sample independently) | Transformers, RNNs, variable-length sequences |

**BatchNorm** computes statistics across the batch: each feature has a mean and variance computed
over all samples. This creates a dependency between samples in a batch.

**LayerNorm** computes statistics across features: each sample is normalized independently.
No cross-sample dependency, which is critical for autoregressive models.

```
Input shape: (batch, features)

BatchNorm normalizes along axis=0 (down columns):
  [x11  x12  x13]       Each column gets its own mean/var
  [x21  x22  x23]  -->  computed from all rows
  [x31  x32  x33]

LayerNorm normalizes along axis=-1 (across rows):
  [x11  x12  x13]  -->  Each row gets its own mean/var
  [x21  x22  x23]  -->  computed from all columns
  [x31  x32  x33]  -->  independent per sample
```

In [ ]:
torch.manual_seed(42)

batch_size, d = 8, 64
x = torch.randn(batch_size, d) * 5 + 3

# LayerNorm: normalize each sample across features
ln_out = LayerNorm(d)(x)

# BatchNorm: normalize each feature across the batch
bn = nn.BatchNorm1d(d)
bn.eval()  # use batch stats, not running stats
bn.train()
bn_out = bn(x)

print("=== LayerNorm (per-sample normalization) ===")
print(f"  Per-sample means (should be ~0): {[f'{m:.4f}' for m in ln_out.mean(dim=-1).tolist()]}")
print(f"  Per-feature means (not constrained): {ln_out.mean(dim=0)[:5].tolist()}")

print("\n=== BatchNorm (per-feature normalization) ===")
print(f"  Per-feature means (should be ~0): {[f'{m:.4f}' for m in bn_out.mean(dim=0)[:8].tolist()]}")
print(f"  Per-sample means (not constrained): {bn_out.mean(dim=-1)[:5].tolist()}")

## 8. Manual BatchNorm Implementation for Comparison

To solidify the difference, let us implement BatchNorm manually.

In [ ]:
def manual_batchnorm(x, eps=1e-5):
    """BatchNorm: normalize each feature across the batch dimension."""
    # x shape: (batch, features)
    mean = x.mean(dim=0, keepdim=True)   # mean per feature, shape (1, features)
    var = x.var(dim=0, keepdim=True)      # var per feature, shape (1, features)
    return (x - mean) / torch.sqrt(var + eps)

def manual_layernorm(x, eps=1e-5):
    """LayerNorm: normalize each sample across the feature dimension."""
    # x shape: (batch, features)
    mean = x.mean(dim=-1, keepdim=True)  # mean per sample, shape (batch, 1)
    var = x.var(dim=-1, keepdim=True)     # var per sample, shape (batch, 1)
    return (x - mean) / torch.sqrt(var + eps)

torch.manual_seed(42)
x = torch.randn(8, 64) * 5 + 3

bn_manual = manual_batchnorm(x)
ln_manual = manual_layernorm(x)

print("Manual BatchNorm:")
print(f"  Feature 0 mean across batch: {bn_manual[:, 0].mean().item():.6f}")
print(f"  Feature 0 var across batch:  {bn_manual[:, 0].var().item():.4f}")

print("\nManual LayerNorm:")
print(f"  Sample 0 mean across features: {ln_manual[0].mean().item():.6f}")
print(f"  Sample 0 var across features:  {ln_manual[0].var().item():.4f}")

In [ ]:
# When is each appropriate?
print("When to use BatchNorm:")
print("  - CNNs with fixed-size batches")
print("  - Computer vision tasks")
print("  - When batch statistics are meaningful")
print()
print("When to use LayerNorm:")
print("  - Transformers and attention-based models")
print("  - Recurrent networks (RNNs, LSTMs)")
print("  - Variable-length sequences")
print("  - Small batch sizes or batch_size=1 (inference)")
print("  - When samples must be normalized independently")

## 9. Effect on Training: Deep Linear Stack

One of the most dramatic effects of normalization is preventing activation explosion
or collapse in deep networks. Let us stack 10 linear layers and track what happens
to activation statistics with and without LayerNorm.

In [ ]:
torch.manual_seed(42)

d = 128
n_layers = 10

# Build two stacks: with and without LayerNorm
layers_no_norm = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])
layers_with_norm = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])
norms = nn.ModuleList([LayerNorm(d) for _ in range(n_layers)])

# Copy weights so the comparison is fair
for i in range(n_layers):
    layers_with_norm[i].weight.data = layers_no_norm[i].weight.data.clone()
    layers_with_norm[i].bias.data = layers_no_norm[i].bias.data.clone()

x = torch.randn(16, d)

# Track stats through each layer -- without normalization
means_no_norm, stds_no_norm = [], []
h = x.clone()
for i, layer in enumerate(layers_no_norm):
    h = torch.relu(layer(h))
    means_no_norm.append(h.mean().item())
    stds_no_norm.append(h.std().item())

# Track stats through each layer -- with normalization
means_with_norm, stds_with_norm = [], []
h = x.clone()
for i in range(n_layers):
    h = layers_with_norm[i](h)
    h = norms[i](h)
    h = torch.relu(h)
    means_with_norm.append(h.mean().item())
    stds_with_norm.append(h.std().item())

print("Layer | Without Norm (mean, std) | With Norm (mean, std)")
print("-" * 60)
for i in range(n_layers):
    print(f"  {i+1:2d}  |  mean={means_no_norm[i]:10.4f}  std={stds_no_norm[i]:10.4f}  |  "
          f"mean={means_with_norm[i]:.4f}  std={stds_with_norm[i]:.4f}")

## 10. Plot: Activation Statistics Across Layers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

layer_indices = list(range(1, n_layers + 1))

axes[0].plot(layer_indices, means_no_norm, 'o-', color='steelblue', label='Without LayerNorm')
axes[0].plot(layer_indices, means_with_norm, 's-', color='coral', label='With LayerNorm')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Activation Mean')
axes[0].set_title('Mean of Activations per Layer')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(layer_indices, stds_no_norm, 'o-', color='steelblue', label='Without LayerNorm')
axes[1].plot(layer_indices, stds_with_norm, 's-', color='coral', label='With LayerNorm')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Activation Std')
axes[1].set_title('Std of Activations per Layer')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Without normalization, statistics can drift or explode.")
print("With LayerNorm, activations stay in a well-behaved range at every layer.")

## 11. Gradient Flow Through Deep Networks

Normalization also helps gradients flow backward. Without it, gradients can explode or
vanish as they propagate through many layers. Let us compare gradient norms with
and without LayerNorm.

In [ ]:
torch.manual_seed(42)

d = 128
n_layers = 10

def build_stack(use_norm):
    """Build a deep stack and return intermediate activations for gradient analysis."""
    layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])
    norms = nn.ModuleList([LayerNorm(d) for _ in range(n_layers)]) if use_norm else None
    return layers, norms

def forward_and_backward(layers, norms, x):
    """Forward pass, then backward. Return gradient norms of each layer's weights."""
    h = x
    for i, layer in enumerate(layers):
        h = layer(h)
        if norms is not None:
            h = norms[i](h)
        h = torch.relu(h)

    loss = h.sum()
    loss.backward()

    grad_norms = []
    for layer in layers:
        grad_norms.append(layer.weight.grad.norm().item())
    return grad_norms

# Without normalization
torch.manual_seed(42)
layers_no, norms_no = build_stack(use_norm=False)
x = torch.randn(16, d)
grads_no_norm = forward_and_backward(layers_no, norms_no, x)

# With normalization
torch.manual_seed(42)
layers_yes, norms_yes = build_stack(use_norm=True)
x = torch.randn(16, d)
grads_with_norm = forward_and_backward(layers_yes, norms_yes, x)

print("Layer | Grad Norm (no norm) | Grad Norm (with norm)")
print("-" * 55)
for i in range(n_layers):
    print(f"  {i+1:2d}  |  {grads_no_norm[i]:14.6f}    |  {grads_with_norm[i]:14.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.semilogy(layer_indices, grads_no_norm, 'o-', color='steelblue', label='Without LayerNorm')
ax.semilogy(layer_indices, grads_with_norm, 's-', color='coral', label='With LayerNorm')
ax.set_xlabel('Layer (from input to output)')
ax.set_ylabel('Gradient Norm (log scale)')
ax.set_title('Gradient Norms Across Layers')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("With LayerNorm, gradient norms stay more uniform across layers.")
print("Without it, gradients can vary by orders of magnitude.")

## 12. Pre-Norm vs. Post-Norm

In transformer architectures, there are two ways to place LayerNorm:

**Post-norm** (original Transformer -- Vaswani et al. 2017):
```
x -> SubLayer -> Add(x) -> LayerNorm -> output
```

**Pre-norm** (GPT-2, most modern LLMs):
```
x -> LayerNorm -> SubLayer -> Add(x) -> output
```

Pre-norm is more stable because the residual connection carries the original (unnormalized)
signal, and normalization happens before each sublayer. This makes training easier,
especially for very deep models.

In [ ]:
class PostNormBlock(nn.Module):
    """Post-norm: sublayer -> add -> norm (original Transformer)"""
    def __init__(self, d):
        super().__init__()
        self.linear = nn.Linear(d, d)
        self.norm = LayerNorm(d)

    def forward(self, x):
        h = torch.relu(self.linear(x))  # sublayer
        return self.norm(x + h)          # add & norm


class PreNormBlock(nn.Module):
    """Pre-norm: norm -> sublayer -> add (GPT-2 style)"""
    def __init__(self, d):
        super().__init__()
        self.linear = nn.Linear(d, d)
        self.norm = LayerNorm(d)

    def forward(self, x):
        h = self.norm(x)                 # norm first
        h = torch.relu(self.linear(h))   # sublayer
        return x + h                     # residual add


print("Post-norm: x -> ReLU(Linear(x)) -> x + h -> LayerNorm")
print("Pre-norm:  x -> LayerNorm -> ReLU(Linear(h)) -> x + h")

In [ ]:
torch.manual_seed(42)

d = 128
n_blocks = 10
x = torch.randn(16, d)

# Post-norm stack
post_blocks = nn.ModuleList([PostNormBlock(d) for _ in range(n_blocks)])
h_post = x.clone()
post_stds = []
for block in post_blocks:
    h_post = block(h_post)
    post_stds.append(h_post.std().item())

# Pre-norm stack
torch.manual_seed(42)
pre_blocks = nn.ModuleList([PreNormBlock(d) for _ in range(n_blocks)])
h_pre = x.clone()
pre_stds = []
for block in pre_blocks:
    h_pre = block(h_pre)
    pre_stds.append(h_pre.std().item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, n_blocks+1), post_stds, 'o-', color='steelblue', label='Post-norm')
ax.plot(range(1, n_blocks+1), pre_stds, 's-', color='coral', label='Pre-norm')
ax.set_xlabel('Block')
ax.set_ylabel('Activation Std')
ax.set_title('Pre-norm vs Post-norm: Activation Scale Across Blocks')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Pre-norm allows activations to grow gradually (residual accumulation),")
print("while post-norm constrains each block's output. Pre-norm is more stable")
print("for very deep stacks and is used by GPT-2, GPT-3, and most modern LLMs.")

## 13. RMSNorm: A Simpler Alternative

**RMSNorm** (Root Mean Square Normalization) simplifies LayerNorm by removing the mean
subtraction step. Instead of centering and scaling, it only divides by the RMS of the input:

$$\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i$$

Where $\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2}$

RMSNorm is used in LLaMA, Gemma, and other modern architectures because:
- It is computationally cheaper (no mean computation needed)
- It often performs just as well in practice
- It has fewer operations, which is better for hardware efficiency

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        # RMS = sqrt(mean(x^2))
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.g * (x / rms)

print("RMSNorm has no bias parameter and no mean subtraction.")
print(f"RMSNorm parameters: {sum(p.numel() for p in RMSNorm(64).parameters())} (only gamma)")
print(f"LayerNorm parameters: {sum(p.numel() for p in LayerNorm(64).parameters())} (gamma + beta)")

In [ ]:
# Compare LayerNorm vs RMSNorm outputs
torch.manual_seed(42)
x = torch.randn(4, 64) * 5 + 3

ln = LayerNorm(64)
rms = RMSNorm(64)

ln_out = ln(x)
rms_out = rms(x)

print("=== LayerNorm ===")
print(f"  mean: {ln_out.mean(dim=-1).tolist()}")
print(f"  std:  {[f'{s:.4f}' for s in ln_out.std(dim=-1).tolist()]}")

print("\n=== RMSNorm ===")
print(f"  mean: {[f'{m:.4f}' for m in rms_out.mean(dim=-1).tolist()]}")
print(f"  std:  {[f'{s:.4f}' for s in rms_out.std(dim=-1).tolist()]}")

print("\nNote: RMSNorm does NOT center the output (mean is not necessarily 0).")
print("It only normalizes the scale by dividing by the RMS value.")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(x.detach().numpy().flatten(), bins=50, alpha=0.7, color='gray', edgecolor='black', linewidth=0.5)
axes[0].set_title('Original Input')
axes[0].set_xlabel('Value')

axes[1].hist(ln_out.detach().numpy().flatten(), bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
axes[1].set_title('After LayerNorm')
axes[1].set_xlabel('Value')

axes[2].hist(rms_out.detach().numpy().flatten(), bins=50, alpha=0.7, color='coral', edgecolor='black', linewidth=0.5)
axes[2].set_title('After RMSNorm')
axes[2].set_xlabel('Value')

plt.tight_layout()
plt.show()

## 14. Ablation: Effect of Epsilon

The $\epsilon$ parameter prevents division by zero when the variance is very small.
Let us test different values and see how they affect the output, especially
on edge-case inputs.

In [ ]:
eps_values = [1e-1, 1e-3, 1e-5, 1e-8]

# Normal input
torch.manual_seed(42)
x_normal = torch.randn(4, 64)

# Near-constant input (very small variance)
x_constant = torch.ones(4, 64) + torch.randn(4, 64) * 1e-7

print("=== Normal Input (std ~ 1) ===")
print(f"  Input variance: {x_normal.var(dim=-1).mean().item():.6f}")
print()
for eps in eps_values:
    ln = LayerNorm(64, eps=eps)
    out = ln(x_normal)
    print(f"  eps={eps:.0e}: output std={out.std(dim=-1).mean().item():.6f}, "
          f"max={out.abs().max().item():.4f}")

print(f"\n=== Near-Constant Input (std ~ 1e-7) ===")
print(f"  Input variance: {x_constant.var(dim=-1).mean().item():.2e}")
print()
for eps in eps_values:
    ln = LayerNorm(64, eps=eps)
    out = ln(x_constant)
    has_nan = torch.isnan(out).any().item()
    has_inf = torch.isinf(out).any().item()
    print(f"  eps={eps:.0e}: output std={out.std(dim=-1).mean().item():.6f}, "
          f"max={out.abs().max().item():.4f}, NaN={has_nan}, Inf={has_inf}")

In [ ]:
# Visualize epsilon effect on near-constant input
fig, axes = plt.subplots(1, len(eps_values), figsize=(16, 3))

for i, eps in enumerate(eps_values):
    ln = LayerNorm(64, eps=eps)
    out = ln(x_constant)
    vals = out.detach().numpy().flatten()
    axes[i].hist(vals, bins=40, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'eps = {eps:.0e}')
    axes[i].set_xlabel('Value')

plt.suptitle('LayerNorm Output on Near-Constant Input (varying epsilon)', y=1.02)
plt.tight_layout()
plt.show()

print("Smaller epsilon amplifies tiny differences in near-constant inputs.")
print("The default eps=1e-5 provides a good balance for most cases.")
print("Too large eps (1e-1) suppresses normalization; too small risks instability.")

## 15. Key Insights

**LayerNorm is essential for transformer training stability.** Here is a summary of what we covered:

1. **LayerNorm normalizes each sample independently** across the feature dimension, making it
   ideal for sequence models where batch statistics are unreliable or unavailable.

2. **The formula is simple**: subtract the mean, divide by the standard deviation, then apply
   learnable scale ($\gamma$) and shift ($\beta$) parameters.

3. **LayerNorm vs BatchNorm**: BatchNorm normalizes across the batch (each feature independently),
   while LayerNorm normalizes across features (each sample independently). Transformers use LayerNorm.

4. **Pre-norm vs Post-norm**: Modern LLMs (GPT-2, GPT-3, LLaMA) use pre-norm placement,
   which puts LayerNorm *before* each sublayer for more stable training.

5. **RMSNorm** simplifies LayerNorm by removing mean subtraction. It is used in LLaMA and other
   efficient architectures with comparable performance.

6. **Gradient flow**: Normalization keeps gradient norms more uniform across layers,
   preventing vanishing and exploding gradients.

7. **Epsilon** ($\epsilon$) is a small constant for numerical stability. The default value of
   $10^{-5}$ works well in practice.

In [ ]:
# Final summary: LayerNorm in one line
print("LayerNorm in one line:")
print("  output = gamma * (x - mean) / sqrt(var + eps) + beta")
print()
print("Where:")
print("  - mean and var are computed per-sample across the feature dimension")
print("  - gamma (scale) is initialized to 1")
print("  - beta (shift) is initialized to 0")
print("  - eps = 1e-5 for numerical stability")